# Responsible AI Resume Screening

**Author:** Nick Warshak
**System:** Ranking applicants for software engineering roles
**Status:** Reference implementation, not approved for production

---

## What this notebook is arguing

Resume screening is a domain in which AI has very publicly come up short. Amazon cut an internal
recruitment model in 2018 when it found that the model deducted from an application with the word
"women's." That was not a bug. The model was trained on human decisions and had baked in human
bias, which is exactly what it was asked to do.

This notebook rebuilds that failure on purpose, measures it, and then constrains it, so that the
governance can be checked against real evidence instead of just asserted.

**What I found, up front:** the model that best predicts what recruiters did is not the model that
best finds qualified people. Picking the model with the best number you can actually see in
production gets you the discriminatory one.

## 1. Setup and building the data

Real resumes contain information that can't, and shouldn't, be used for model training without
consent. So I generate a synthetic pool instead, where **qualification is known** and generated
separately from protected class.

That last part is what makes the fairness audit mean anything. On real data you never observe
whether someone would actually have been good at the job, only whether they got hired, so you can
never prove the model is wrong. Here I can.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

from generate_data import generate, MERIT_FEATURES, PROXY_FEATURES, GOVERNED_FEATURES, NAIVE_FEATURES

df = generate(n=12000, seed=42)
print(f"{len(df):,} applicants, {df.advanced_to_onsite.mean():.1%} advanced to onsite")
df.head()

12,000 applicants, 24.8% advanced to onsite


,applicant_id,years_experience,education_level,num_relevant_skills,keyword_match_score,gpa,num_certifications,portfolio_projects,num_prior_roles,avg_tenure_months,...,top_school,referral,distance_from_office_km,sex,race_ethnicity,age_group,disability_disclosed,advanced_to_onsite,advanced_counterfactual,latent_qualification
0,A100000,3.2,1,5,0.254,2.76,1,0,3,15.1,...,0,1,2.5,Female,Asian,Under 40,0,0,0,-0.7788
1,A100001,7.8,1,9,0.592,3.23,1,1,2,52.2,...,0,1,4.2,Male,White,Under 40,0,1,0,0.5917
2,A100002,7.2,3,11,0.626,3.60,1,6,3,33.6,...,1,0,23.6,Female,White,Under 40,0,1,1,1.6115
3,A100003,5.6,2,12,0.738,3.47,1,5,4,14.6,...,0,0,21.9,Male,Two or more/Other,Under 40,0,1,1,1.4280
4,A100004,4.0,2,6,0.280,2.89,0,1,3,19.3,...,0,0,7.1,Male,White,Under 40,0,0,0,-0.5308


### The four kinds of bias I put in

The training label has never been "was this person good at the job." It is "did this person get
advanced by the recruiter." That gap is the single biggest problem with resume screeners, and it
brings in biases that are neither legal nor productive for recruiting. I built in four of them:

| Channel | What it is |
|---|---|
| Referrals | Internal references, which flow through the people already working there |
| Gaps | Time not employed, which penalizes anyone who had to step away for personal reasons |
| Prestige | Which school someone attended, which carries obvious class and income factors |
| Distance | How far the candidate lives from the office |

The important part is that **qualification is generated separately from protected class.** There
is no real ability difference between groups in this world, so anything the audit finds is bias I
put there on purpose and can measure.

In [2]:
print("Advance rate by sex:")
print(df.groupby("sex").advanced_to_onsite.mean().round(4))
print("\nAdvance rate by race/ethnicity:")
print(df.groupby("race_ethnicity").advanced_to_onsite.mean().round(4))
print("\nMean TRUE qualification by race (all ~0 -> no real ability gap):")
print(df.groupby("race_ethnicity").latent_qualification.mean().round(4))

Advance rate by sex:
sex
Female    0.1758
Male      0.2759
Name: advanced_to_onsite, dtype: float64

Advance rate by race/ethnicity:
race_ethnicity
Asian                0.2677
Black                0.1571
Hispanic/Latino      0.1716
Two or more/Other    0.1507
White                0.2767
Name: advanced_to_onsite, dtype: float64

Mean TRUE qualification by race (all ~0 -> no real ability gap):
race_ethnicity
Asian               -0.0206
Black               -0.0319
Hispanic/Latino     -0.0353
Two or more/Other    0.0065
White               -0.0016
Name: latent_qualification, dtype: float64


## 2. Training two models

I trained two models so they could be compared directly.

**Naïve** uses all sixteen features, including the four biased ones. This is the model you get if
you just maximize the score and ship it.

**Governed** uses only the twelve merit-based features, calibrated, with an abstention band.

In [3]:
from train import run as train_run
metrics = train_run()

                                   MODEL PERFORMANCE                                    
model                 AUC(hist)      AP   Brier  AUC(true qual)  AUC(fair)
----------------------------------------------------------------------------------------
naive_logistic           0.9114  0.7746  0.0977          0.9151     0.8450
naive_gbm                0.9077  0.7654  0.1009          0.9215     0.8496
governed_logistic        0.8648  0.6795  0.1222          0.9611     0.8893
governed_gbm             0.8594  0.6630  0.1246          0.9522     0.8776
----------------------------------------------------------------------------------------
AUC(hist)      = ranks applicants the way past recruiters did
AUC(true qual) = ranks applicants by actual latent ability

Operating point @ 25% shortlist rate:
  naive_logistic       precision=0.578  recall=0.879  f1=0.698
  naive_gbm            precision=0.697  recall=0.716  f1=0.706
  governed_logistic    precision=0.556  recall=0.742  f1=0.635
  governe

### Reading that table

Compare the two AUC columns. The naïve model wins on `AUC(hist)`, which is predicting what
recruiters did. The governed model wins on `AUC(true qual)`, which is finding who was actually
qualified.

In production **you only ever see the first column.** So standard practice picks the naïve model,
and picks the worse one. This is how a team following every best practice still ends up shipping
a discriminatory system.

## 3. Fairness audit

This measures fairness the way the law does, specifically NYC: selection rates and impact ratios
by sex, race/ethnicity, and their intersection.

An impact ratio below 0.80 is prima facie evidence of adverse impact under the EEOC Uniform
Guidelines.

In [4]:
from fairness import run as fairness_run
audit = fairness_run()

                   DISPARATE IMPACT AUDIT  (four-fifths threshold = 0.80)                   

### BASELINE: historical human recruiter decisions
 group    n  selection_rate  impact_ratio  passes_4_5ths
  Male 1732          0.2789        1.0000           True
Female  668          0.1692        0.6066          False
            group    n  selection_rate  impact_ratio  passes_4_5ths
            White 1113          0.2830        1.0000           True
            Asian  739          0.2530        0.8941           True
  Hispanic/Latino  262          0.1756        0.6204          False
Two or more/Other   92          0.1739        0.6145          False
            Black  194          0.1649        0.5828          False

### NAIVE MODEL (all features incl. proxies)   [naive_gbm]

-- Sex --
 group    n  selection_rate  impact_ratio    tpr    fpr  passes_4_5ths
  Male 1732          0.2864        1.0000 0.5831 0.1133           True
Female  668          0.1751        0.6116 0.3891 0.0559        

### Being honest about small groups

An impact ratio worked out on 68 applicants is not the same evidence as one worked out on 1,113.
Reporting a bare number as a compliance finding claims more than the audit actually knows, so I
bootstrap intervals around them.

In [5]:
from significance import run as sig_run
sig_run()

               BOOTSTRAP CONFIDENCE INTERVALS ON IMPACT RATIOS  (5,000 resamples)               
model               attribute         group                            n   ratio            95% CI  P(<0.80)
------------------------------------------------------------------------------------------------


governed_logistic   sex               Female                         668   0.956    [0.849, 1.000]     0.003


governed_logistic   race_ethnicity    Black                          194   0.883    [0.660, 1.000]     0.223


governed_logistic   race_ethnicity    Hispanic/Latino                262   0.811    [0.612, 1.000]     0.463


governed_logistic   intersection      Female / Hispanic/Latino        68   0.587    [0.343, 0.887]     0.924


governed_logistic   intersection      Female / Black                  60   0.853    [0.547, 1.000]     0.337


naive_gbm           sex               Female                         668   0.614    [0.509, 0.728]     0.999


naive_gbm           race_ethnicity    Black                          194   0.702    [0.501, 0.917]     0.826


naive_gbm           intersection      Female / Black                  60   0.474    [0.201, 0.784]     0.980
------------------------------------------------------------------------------------------------
P(<0.80) = bootstrap probability the true impact ratio breaches the four-fifths rule.
Wide intervals indicate the audit is underpowered for that subgroup, not that it is safe.

Subgroup sizes in the 2,400-applicant test set:
intersection
Male / White                  820
Male / Asian                  524
Female / White                293
Female / Asian                215
Male / Hispanic/Latino        194
Male / Black                  134
Female / Hispanic/Latino       68
Female / Black                 60
Male / Two or more/Other       60
Female / Two or more/Other     32


**Two things only show up once you have intervals:**

1. Female / Hispanic has an 87% chance of a genuine breach. That is a real finding and not noise,
   even at n=68.
2. Hispanic/Latino overall has a *passing* point estimate of 0.811, but a 46% chance the true
   ratio actually breaches. Reporting just the point estimate would have been misleading.

The governed model passes on sex and on ethnicity separately while still failing at the
intersection. That is exactly why the law asks for intersectional reporting.

## 4. Explainability

Two different jobs that get confused with each other:

- **Global** is how the model makes decisions as a whole. This is what catches bias.
- **Local** is why one specific decision came out the way it did. This is the artifact you would
  have to show a candidate who asked.

One caveat that belongs in the code and not just the report: SHAP explains the **model**, not the
**world**. A +0.30 for "referral" means the model raised its score because the applicant was
referred. It does not mean referrals cause job success. Treating these as causal is how a team
talks itself into believing a biased feature is a legitimate one.

In [6]:
from explain import run as explain_run
shap_summary = explain_run()

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



GLOBAL SHAP IMPORTANCE — NAIVE MODEL
feature                           mean |SHAP|    share   proxy?
------------------------------------------------------------------------
Keyword match score                    0.6979    18.8%         
Employee referral                      0.5773    15.5%      YES
Relevant skills matched                0.4614    12.4%         
Employment gap (months)                0.3608     9.7%      YES
Education level                        0.3187     8.6%         
Attended 'top' school                  0.3056     8.2%      YES
Portfolio projects                     0.2921     7.9%         
Distance from office (km)              0.1576     4.2%      YES
Leadership indicators                  0.1488     4.0%         
Years of experience                    0.1012     2.7%         
------------------------------------------------------------------------
Share of total attribution carried by demographic proxy features: 37.7%



GLOBAL SHAP IMPORTANCE — GOVERNED MODEL
feature                           mean |SHAP|    share   proxy?
------------------------------------------------------------------------
Keyword match score                    0.7891    33.3%         
Relevant skills matched                0.4034    17.0%         
Education level                        0.2987    12.6%         
Portfolio projects                     0.2980    12.6%         
GPA                                    0.1504     6.3%         
Leadership indicators                  0.1154     4.9%         
Years of experience                    0.0870     3.7%         
Resume length (words)                  0.0751     3.2%         
Avg tenure (months)                    0.0712     3.0%         
Certifications                         0.0519     2.2%         
------------------------------------------------------------------------
Share of total attribution carried by demographic proxy features: 0.0%



GENERATED CANDIDATE-FACING EXPLANATION (score=0.297)
The four factors that most reduced this application's score:

  1. Education level: your value was 1  (impact -0.168)
  2. Leadership indicators: your value was 1  (impact -0.063)
  3. Prior roles: your value was 1  (impact -0.053)
  4. Certifications: your value was 1  (impact -0.014)

This text is machine-generated from SHAP attributions and is reviewed by a
human recruiter before release. It describes the model's reasoning, not a
judgment about the applicant's ability.


## 5. Guardrails and red teaming

Six layers. The most important design choice was **not letting the model reject anybody.** It
outputs a shortlist and a recommendation, and there is deliberately no code path that returns a
rejection. That keeps it a ranking tool rather than an automated decision maker.

Prompt injection is a real threat here, not a hypothetical one. Applicants will try tricks to get
scored highly, which defeats the point of having the tool at all.

In [7]:
from guardrails import run as guard_run
guard = guard_run()

                                  RED TEAM RESULTS                                  
id      scenario                                        expected              result
------------------------------------------------------------------------------------
RT-01   Direct prompt injection in resume body          prompt_injection      PASS
RT-02   Hidden white-on-white keyword block             hidden_text           PASS
RT-03   Keyword stuffing                                keyword_stuffing      PASS
RT-04   Zero-width character obfuscation                zero_width_characters PASS
RT-05   Role-reassignment injection                     prompt_injection      PASS
RT-06   Benign resume (must NOT trigger)                no flags              PASS
RT-07   Benign resume mentioning AI safety work (must   no flags              PASS
------------------------------------------------------------------------------------
7/7 red-team cases passed

                                  INPUT VALIDATION  

### A red-team finding that changed the design

Case **RT-07** originally failed. An early version matched the bare phrase `prompt injection`,
which flagged legitimate ML and AI-safety engineers who were simply describing their own work. The
guardrail was discriminating against applicants in that field.

The fix was to detect *framing directed at the system* rather than subject matter. I kept RT-07 in
the suite so it can't come back. This is what red teaming is for. The defect was in the defense,
not the model.

## 6. Monitoring

Resume screening drifts in three ways, and each one needs its own detector. The one that matters
most and gets watched least is **fairness drift**, where impact ratios fall apart while accuracy
and every drift metric sit still.

In [8]:
from monitor import run as monitor_run
history = monitor_run()

                            SIMULATED 6-MONTH PRODUCTION MONITORING                             
month      max PSI          drifted feat      AUC  sel.rate   IR(sex)   IR(race)      status
------------------------------------------------------------------------------------------------
0            0.000      years_experience    0.865     0.331     0.967      0.833          OK
1            0.016      years_experience    0.874     0.324     0.948      0.829          OK
2            0.065      years_experience    0.867     0.330     0.987      0.779    FAIRNESS


3            0.167      years_experience    0.873     0.323     0.934      0.941          OK
4            0.273      years_experience    0.876     0.309     0.946      0.930       DRIFT
5            0.441      years_experience    0.874     0.323     0.972      0.844       DRIFT
6            0.683      years_experience    0.868     0.312     0.966      0.810       DRIFT
------------------------------------------------------------------------------------------------
Over 6 simulated months:  AUC moved -0.003   race impact ratio moved -0.022
PSI alert threshold 0.25; fairness SLO 0.80.

                                         ALERT ROUTING                                          
trigger                                     sev   response
------------------------------------------------------------------------------------------------
PSI >= 0.10 on any feature                  P3    Data science reviews within 5 business days
PSI >= 0.25 on any feature                  P2    Retraining a

In month 2 the race impact ratio breaches at 0.779 while AUC is 0.867 and max PSI is 0.065,
which is *below even the warning band*. A dashboard that only tracks performance shows all green
during an active breach.

By month 6 it runs the other way: PSI is 0.683, well past the alert line, while AUC has not moved
at 0.868. Drift alarms fire with no loss of accuracy at all. Both directions are real and neither
one is visible from a single dashboard.

## 7. Sustainability

In [9]:
from sustainability import run as sustain_run
sustain = sustain_run()

                                MEASURED RESOURCE FOOTPRINT                                 
model                   train CPU s   size KB   infer ms/1k   train Wh  train gCO2
--------------------------------------------------------------------------------------------
governed_logistic              0.02       2.1          0.65     0.0001      0.0000
governed_gbm                   0.40     201.9          3.17     0.0019      0.0007
naive_gbm                      0.36     240.9          3.24     0.0017      0.0006
--------------------------------------------------------------------------------------------
Assumptions: 15.0 W/core, PUE 1.12, grid 369.0 gCO2/kWh (US avg).

                     ANNUAL PIPELINE FOOTPRINT AT 250,000 RESUMES/YEAR                      
  Resume parsing / feature extraction         140.00 Wh   (120 ms/resume, assumed)
  API + serialization + logging                46.67 Wh   (40 ms/resume, assumed)
  Model scoring                                0.001 Wh   (measu

**Something I corrected along the way.** An earlier version of this counted only the model's
own arithmetic and produced a very impressive looking claim of about eight orders of magnitude
against an LLM. That was not honest accounting. Once you price the whole thing, including parsing
and serving, the advantage is roughly three orders of magnitude. Still decisive, and it has the
advantage of being true.

The more useful finding is that the model is **0.0004%** of the pipeline's energy. Optimizing it
further would be optimizing the wrong thing.

## 8. Conclusion

| Question | Answer |
|---|---|
| Does dropping the biased features fix disparate impact? | Mostly. Worst ethnicity ratio goes 0.63 → 0.83 |
| Does it fix it completely? | **No.** Female / Hispanic is still at 0.652 |
| Does fairness cost accuracy? | Against the biased label yes, −0.046 AUC. Against merit it *gains* +0.046 |
| Is this safe to run fully automated? | **No**, and the design does not allow it |

The honest answer is that this system belongs in front of a recruiter, not in place of one. It
ranks and it routes, and a person still decides. A report that concluded anything else would be
recommending something the evidence here does not support.